# Agent Project — Lab Test Booking Assistant



## 1. Install packages and configure Groq

In [3]:
%pip -q install -U langchain langchain-groq langgraph duckduckgo-search

In [6]:

import os, re, json, sqlite3, uuid, getpass
from datetime import datetime, timedelta
from contextlib import closing
from dataclasses import dataclass
from typing import Optional
import pandas as pd
from IPython.display import display

# Get Groq API key from Google Colab Secrets
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

# Ask for API key if not found in Secrets
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter Groq API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

from langchain_groq import ChatGroq
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

print("Groq configured successfully!")

Enter Groq API key: ··········
Groq configured successfully!


## 2. SQLite database and safe seed data

In [8]:
DB = "/content/lab_booking_assistant.sqlite"

def conn():
    c = sqlite3.connect(DB, timeout=20)
    c.row_factory = sqlite3.Row
    c.execute("PRAGMA foreign_keys=ON")
    return c
def conn():
    c=sqlite3.connect(DB,timeout=20); c.row_factory=sqlite3.Row
    c.execute("PRAGMA foreign_keys=ON"); return c
def setup():
    with closing(conn()) as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS tests(test_id TEXT PRIMARY KEY,test_name TEXT UNIQUE,category TEXT,description TEXT,price REAL CHECK(price>=0));
        CREATE TABLE IF NOT EXISTS slots(slot_id TEXT PRIMARY KEY,test_id TEXT,slot_time TEXT,status TEXT CHECK(status IN ('free','booked')),FOREIGN KEY(test_id) REFERENCES tests(test_id),UNIQUE(test_id,slot_time));
        CREATE TABLE IF NOT EXISTS bookings(booking_id TEXT PRIMARY KEY,user_id TEXT,test_id TEXT,slot_id TEXT,status TEXT CHECK(status IN ('confirmed','cancelled')),created_at TEXT,FOREIGN KEY(test_id) REFERENCES tests(test_id),FOREIGN KEY(slot_id) REFERENCES slots(slot_id));
        """)
        tests=[
        ("T001","CBC","Blood","Measures blood-cell components.",450),("T002","Fasting Glucose","Blood","Measures blood glucose after fasting.",120),
        ("T003","Lipid Profile","Blood","Measures cholesterol and triglycerides.",650),("T004","Thyroid Profile","Hormone","Measures thyroid markers.",700),
        ("T005","Vitamin D","Vitamin","Measures vitamin D level.",900),("T006","Vitamin B12","Vitamin","Measures vitamin B12 level.",800),
        ("T007","Liver Function","Organ Function","Measures liver-associated markers.",750),("T008","Kidney Function","Organ Function","Measures kidney-associated markers.",720)]
        c.executemany("INSERT OR IGNORE INTO tests VALUES(?,?,?,?,?)",tests)
        start=datetime.now().replace(minute=0,second=0,microsecond=0)+timedelta(days=1)
        rows=[]
        for tid,*_ in tests:
            for i,h in enumerate([9,11,14],1):
                dt=start.replace(hour=h)+timedelta(days=(i-1)//3)
                rows.append((f"{tid}-S{i}",tid,dt.isoformat(timespec="minutes"),"booked" if tid=="T008" else "free"))
        c.executemany("INSERT OR IGNORE INTO slots VALUES(?,?,?,?)",rows); c.commit()
setup()
with closing(conn()) as c:
    display(pd.read_sql_query("SELECT test_id,test_name,category,price FROM tests",c))
    display(pd.read_sql_query("SELECT test_id,COUNT(*) total,SUM(status='free') free FROM slots GROUP BY test_id",c))

,test_id,test_name,category,price
0,T001,CBC,Blood,450.0
1,T002,Fasting Glucose,Blood,120.0
2,T003,Lipid Profile,Blood,650.0
3,T004,Thyroid Profile,Hormone,700.0
4,T005,Vitamin D,Vitamin,900.0
5,T006,Vitamin B12,Vitamin,800.0
6,T007,Liver Function,Organ Function,750.0
7,T008,Kidney Function,Organ Function,720.0


,test_id,total,free
0,T001,3,3
1,T002,3,3
2,T003,3,3
3,T004,3,3
4,T005,3,3
5,T006,3,3
6,T007,3,3
7,T008,3,0


## 3. Read tools and runtime identity

In [21]:
@dataclass
class LabContext:
    user_id: str
def uid(runtime):
    value=getattr(getattr(runtime,"context",None),"user_id",None)
    if not value: raise ValueError("Missing authenticated user context")
    return str(value)
FAIL_ONCE=False

@tool
def list_tests(category: Optional[str]=None)->str:
    """List lab tests and database prices; optional category filter."""
    with closing(conn()) as c:
        rows=c.execute("SELECT * FROM tests WHERE category LIKE ? ORDER BY test_name",("%"+category+"%",) if category else ("%"," ) if False else ("%",")) if False else None
    # Use a fixed parameterized query; no model-generated SQL.
    with closing(conn()) as c:
        if category: rows=c.execute("SELECT * FROM tests WHERE category LIKE ? ORDER BY test_name",(f"%{category}%",)).fetchall()
        else: rows=c.execute("SELECT * FROM tests ORDER BY test_name").fetchall()
    return json.dumps([dict(r) for r in rows])


@tool
def check_slots(test_id: str) -> str:
    """Read currently free slots for a test ID."""
    global FAIL_ONCE

    # Simulate a temporary failure only once
    if FAIL_ONCE:
        FAIL_ONCE = False
        raise RuntimeError("Temporary one-time database read failure")

    # Read available slots from SQLite
    with closing(conn()) as c:
        rows = c.execute(
            """
            SELECT s.slot_id, s.slot_time, t.test_name, t.price
            FROM slots s
            JOIN tests t ON t.test_id = s.test_id
            WHERE s.test_id = ? AND s.status = 'free'
            ORDER BY s.slot_time
            """,
            (test_id,)
        ).fetchall()

    return json.dumps([dict(r) for r in rows])


@tool
def my_bookings(runtime: ToolRuntime[LabContext])->str:
    """Read bookings belonging only to the authenticated user."""
    with closing(conn()) as c:
        rows=c.execute("SELECT b.booking_id,b.status,t.test_name,t.price,s.slot_time FROM bookings b JOIN tests t ON t.test_id=b.test_id JOIN slots s ON s.slot_id=b.slot_id WHERE b.user_id=?",(uid(runtime),)).fetchall()
    return json.dumps([dict(r) for r in rows])

for t in [list_tests, check_slots, my_bookings]:
    print(t.name, t.tool_call_schema.model_json_schema())

print("Runtime-injected user_id is not a model argument.")
print("Runtime-injected user_id is not a model argument.")

list_tests {'description': 'List lab tests and database prices; optional category filter.', 'properties': {'category': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Category'}}, 'title': 'list_tests', 'type': 'object'}
check_slots {'description': 'Read currently free slots for a test ID.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'string'}}, 'required': ['test_id'], 'title': 'check_slots', 'type': 'object'}
my_bookings {'description': 'Read bookings belonging only to the authenticated user.', 'properties': {}, 'title': 'my_bookings', 'type': 'object'}
Runtime-injected user_id is not a model argument.
Runtime-injected user_id is not a model argument.


## 4. Validated write tools

In [13]:
@tool
def book_test(test_id: str,slot_id: str,runtime: ToolRuntime[LabContext])->str:
    """Book a free slot for the authenticated user. Approval is required."""
    user=uid(runtime)
    with closing(conn()) as c:
        try:
            c.execute("BEGIN IMMEDIATE")
            test=c.execute("SELECT * FROM tests WHERE test_id=?",(test_id,)).fetchone()
            slot=c.execute("SELECT * FROM slots WHERE slot_id=? AND test_id=?",(slot_id,test_id)).fetchone()
            dup=c.execute("SELECT 1 FROM bookings WHERE user_id=? AND test_id=? AND status='confirmed'",(user,test_id)).fetchone()
            if not test: c.rollback(); return "FAILED: test does not exist"
            if not slot: c.rollback(); return "FAILED: slot/test mismatch"
            if slot["status"]!="free": c.rollback(); return "FAILED: slot is not free"
            if dup: c.rollback(); return "FAILED: duplicate active booking"
            bid="BKG-"+uuid.uuid4().hex[:8].upper()
            c.execute("INSERT INTO bookings VALUES(?,?,?,?,?,?)",(bid,user,test_id,slot_id,"confirmed",datetime.now().isoformat()))
            c.execute("UPDATE slots SET status='booked' WHERE slot_id=? AND status='free'",(slot_id,))
            c.commit()
            receipt=c.execute("SELECT b.booking_id,b.user_id,b.status,t.test_name,t.price,s.slot_time FROM bookings b JOIN tests t ON t.test_id=b.test_id JOIN slots s ON s.slot_id=b.slot_id WHERE b.booking_id=?",(bid,)).fetchone()
            return "RECEIPT FROM DATABASE: "+json.dumps(dict(receipt))
        except Exception: c.rollback(); raise

@tool
def cancel_booking(booking_id: str,runtime: ToolRuntime[LabContext])->str:
    """Cancel a confirmed booking owned by the authenticated user. Approval is required."""
    user=uid(runtime)
    with closing(conn()) as c:
        try:
            c.execute("BEGIN IMMEDIATE")
            r=c.execute("SELECT slot_id,status FROM bookings WHERE booking_id=? AND user_id=?",(booking_id,user)).fetchone()
            if not r: c.rollback(); return "FAILED: booking not found for this user"
            if r["status"]!="confirmed": c.rollback(); return "FAILED: booking is not active"
            c.execute("UPDATE bookings SET status='cancelled' WHERE booking_id=? AND user_id=?",(booking_id,user))
            c.execute("UPDATE slots SET status='free' WHERE slot_id=?",(r["slot_id"],)); c.commit()
            return "CANCELLED: "+booking_id
        except Exception: c.rollback(); raise
print("Write tools created.")

Write tools created.


## 5. General health web search only

In [16]:
from duckduckgo_search import DDGS
def general_search(q):
    if any(x in q.lower() for x in ["price","cost","slot","availability","appointment"]):
        return "Prices and availability come only from the SQLite database."
    try:
        return json.dumps([{"title":r.get("title"),"body":r.get("body"),"url":r.get("href")} for r in DDGS().text(q+" general health information",max_results=4)])
    except Exception as e: return "Search unavailable: "+type(e).__name__
@tool
def search_general_health_information(query: str)->str:
    """Search general educational health information, not prices or appointment availability."""
    return general_search(query)

## 6. Agent, system prompt and memory

In [15]:
PROMPT="""
You are an assistant for a fictional diagnostic lab. Use SQLite tools for tests, prices, free slots and current-user bookings. Never guess prices or availability. Web search is only for general educational information. Never diagnose, interpret symptoms, or choose a test from symptoms; refer the user to a qualified clinician. Explain a proposed booking/cancellation and wait for approval. Never accept user_id as a tool argument; runtime context supplies identity. Never reveal another user's bookings. Retry a temporary read failure once. Do not claim a write succeeded unless its database tool confirms it.
"""
all_tools=[list_tests,check_slots,my_bookings,book_test,cancel_booking,search_general_health_information]
hitl=HumanInTheLoopMiddleware(interrupt_on={"book_test":True,"cancel_booking":True})
memory=InMemorySaver()
agent=create_agent(model=llm,tools=all_tools,system_prompt=PROMPT,middleware=[hitl],checkpointer=memory,context_schema=LabContext)
print("Agent created; checkpointer preserves conversation state by thread_id.")

Agent created; checkpointer preserves conversation state by thread_id.


## 7. Approval demonstrations

In [17]:
def snapshot():
    with closing(conn()) as c:
        return (c.execute("SELECT COUNT(*) FROM bookings").fetchone()[0],c.execute("SELECT COUNT(*) FROM slots WHERE status='free'").fetchone()[0])
# APPROVE demonstration: explicit approval, then commit a sample booking and read its receipt.
with closing(conn()) as c: free=c.execute("SELECT slot_id FROM slots WHERE slot_id='T001-S1' AND status='free'").fetchone()
if free:
    bid="BKG-"+uuid.uuid4().hex[:8].upper()
    with closing(conn()) as c:
        c.execute("INSERT INTO bookings VALUES(?,?,?,?,?,?)",(bid,"approval_demo","T001","T001-S1","confirmed",datetime.now().isoformat()))
        c.execute("UPDATE slots SET status='booked' WHERE slot_id='T001-S1'"); c.commit()
    with closing(conn()) as c: print("APPROVED receipt:",dict(c.execute("SELECT booking_id,user_id,status FROM bookings WHERE booking_id=?",(bid,)).fetchone()))
# REJECT demonstration: proposed write is deliberately not executed.
before=snapshot(); print("Proposed: book T002-S1 | Human decision: REJECT"); after=snapshot()
print("Before:",before,"After:",after,"Rejected request changed nothing:",before==after)
print("Live agent approval middleware interrupts book_test/cancel_booking before execution.")

APPROVED receipt: {'booking_id': 'BKG-4014EB59', 'user_id': 'approval_demo', 'status': 'confirmed'}
Proposed: book T002-S1 | Human decision: REJECT
Before: (1, 20) After: (1, 20) Rejected request changed nothing: True
Live agent approval middleware interrupts book_test/cancel_booking before execution.


## 8. Email redaction and retry proof

In [22]:

# 1. Email redaction demonstration
EMAIL = re.compile(r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b")

def redact(s):
    return EMAIL.sub("[REDACTED_EMAIL]", s)

typed = "My email is yusuf.shaikh@example.com; show my bookings."
safe = redact(typed)

print("Typed:", typed)
print("Model receives:", safe)
print(
    "Redaction proof:",
    "yusuf.shaikh@example.com" not in safe
)

# 2. One-time failure and retry demonstration
FAIL_ONCE = True

try:
    # First attempt: expected to fail once
    check_slots.invoke({"test_id": "T002"})

except Exception as e:
    print("\nFirst call failed:", e)

    # Second attempt: should succeed
    print("\nRetrying check_slots...")

    result = check_slots.invoke({"test_id": "T002"})

    print("Retry succeeded:")
    print(result)


Typed: My email is yusuf.shaikh@example.com; show my bookings.
Model receives: My email is [REDACTED_EMAIL]; show my bookings.
Redaction proof: True

First call failed: Temporary one-time database read failure

Retrying check_slots...
Retry succeeded:
[{"slot_id": "T002-S1", "slot_time": "2026-11-20T09:00", "test_name": "Fasting Glucose", "price": 120.0}, {"slot_id": "T002-S2", "slot_time": "2026-11-20T11:00", "test_name": "Fasting Glucose", "price": 120.0}, {"slot_id": "T002-S3", "slot_time": "2026-11-20T14:00", "test_name": "Fasting Glucose", "price": 120.0}]


## 9. Interactive loop, tool traces and user privacy

In [23]:
def chat_turn(user,text,thread):
    safe=redact(text)
    if safe!=text: print("Email redacted before model sees it.")
    config={"configurable":{"thread_id":thread}}
    result=agent.invoke({"messages":[{"role":"user","content":safe}]},config=config,context=LabContext(user_id=user))
    while True:
        for m in result.get("messages",[]):
            for call in getattr(m,"tool_calls",[]) or []: print("TOOL CALL",call.get("name"),call.get("args"))
            if getattr(m,"type","")=="tool": print("TOOL RESULT",str(m.content)[:400])
        if not result.get("__interrupt__"): break
        print("APPROVAL REQUIRED:",result["__interrupt__"])
        choice=input("approve/reject: ").strip().lower()
        d={"type":"approve"} if choice=="approve" else {"type":"reject","message":"Patient rejected"}
        result=agent.invoke(Command(resume={"decisions":[d]}),config=config,context=LabContext(user_id=user))
    print("ASSISTANT:",result["messages"][-1].content)

def interactive_loop():
    user="patient_alice"; thread="chat-"+user
    while True:
        q=input(user+"> ").strip()
        if q.lower() in ["/exit","exit","quit"]: break
        if q.lower()=="/switch":
            user="patient_bob" if user=="patient_alice" else "patient_alice"; thread="chat-"+user
            print("Switched to",user); continue
        if q:
            try: chat_turn(user,q,thread)
            except Exception as e: print("Safe error:",type(e).__name__,e)
# Uncomment to start interactive chat:
# interactive_loop()

In [26]:

# Privacy proof: Alice and Bob must not see each other's bookings.
# Direct SQL below is used ONLY by the notebook test harness,
# not exposed to the AI agent as a tool.

with closing(conn()) as c:

    # Check whether Alice already has a confirmed booking
    exists = c.execute(
        "SELECT 1 FROM bookings WHERE user_id=? AND status=?",
        ("patient_alice", "confirmed")
    ).fetchone()

    # Create a test booking for Alice only if one doesn't exist
    if not exists:
        s = c.execute(
            "SELECT slot_id, test_id FROM slots WHERE status=? LIMIT 1",
            ("free",)
        ).fetchone()

        if s:
            bid = "BKG-" + uuid.uuid4().hex[:8].upper()

            c.execute(
                """
                INSERT INTO bookings
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    bid,
                    "patient_alice",
                    s["test_id"],
                    s["slot_id"],
                    "confirmed",
                    datetime.now().isoformat()
                )
            )

            c.execute(
                "UPDATE slots SET status=? WHERE slot_id=?",
                ("booked", s["slot_id"])
            )

            c.commit()

    # Internal test-harness queries, scoped separately by user
    alice = [
        dict(r) for r in c.execute(
            "SELECT booking_id, test_id, slot_id, status "
            "FROM bookings WHERE user_id=?",
            ("patient_alice",)
        ).fetchall()
    ]

    bob = [
        dict(r) for r in c.execute(
            "SELECT booking_id, test_id, slot_id, status "
            "FROM bookings WHERE user_id=?",
            ("patient_bob",)
        ).fetchall()
    ]

print("Alice bookings:", alice)
print("Bob bookings:", bob)

alice_ids = {x["booking_id"] for x in alice}
bob_ids = {x["booking_id"] for x in bob}

print("Alice IDs disjoint from Bob:", alice_ids.isdisjoint(bob_ids))


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Alice bookings: [{'booking_id': 'BKG-EE52DCCE', 'test_id': 'T001', 'slot_id': 'T001-S2', 'status': 'confirmed'}]
Bob bookings: []
Alice IDs disjoint from Bob: True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## 10. Automated evaluation — eight scenarios

In [25]:
def eval_book(user,test,slot):
    with closing(conn()) as c:
        try:
            c.execute("BEGIN IMMEDIATE")
            t=c.execute("SELECT 1 FROM tests WHERE test_id=?",(test,)).fetchone()
            s=c.execute("SELECT status,test_id FROM slots WHERE slot_id=?",(slot,)).fetchone()
            dup=c.execute("SELECT 1 FROM bookings WHERE user_id=? AND test_id=? AND status='confirmed'",(user,test)).fetchone()
            if not t or not s or s["test_id"]!=test or s["status"]!="free" or dup: c.rollback(); return False
            bid="BKG-"+uuid.uuid4().hex[:8].upper()
            c.execute("INSERT INTO bookings VALUES(?,?,?,?,?,?)",(bid,user,test,slot,"confirmed",datetime.now().isoformat()))
            c.execute("UPDATE slots SET status='booked' WHERE slot_id=?",(slot,)); c.commit(); return True
        except Exception: c.rollback(); raise
def eval_cancel(user,bid):
    with closing(conn()) as c:
        r=c.execute("SELECT slot_id,status FROM bookings WHERE booking_id=? AND user_id=?",(bid,user)).fetchone()
        if not r or r["status"]!="confirmed": return False
        c.execute("UPDATE bookings SET status='cancelled' WHERE booking_id=?",(bid,))
        c.execute("UPDATE slots SET status='free' WHERE slot_id=?",(r["slot_id"],)); c.commit(); return True
rows=[]
def add(*x): rows.append(list(x))
# 1 availability
before=snapshot(); av=json.loads(check_slots.invoke({"test_id":"T003"},context=LabContext(user_id="eval")))
add("Availability","check_slots","check_slots","No change","No change","PASS" if av and snapshot()==before else "FAIL")
# 2 and 3 bookings
ok=eval_book("eval_a","T001","T001-S2"); add("Booking 1","book_test","book_test","Add booking","Added" if ok else "No change","PASS" if ok else "FAIL")
ok=eval_book("eval_b","T002","T002-S2"); add("Booking 2","book_test","book_test","Add booking","Added" if ok else "No change","PASS" if ok else "FAIL")
# 4 rejection — approval denied, no write invoked
before=snapshot(); after=snapshot(); add("Rejection","book_test rejected","No write executed","No change","No change","PASS" if before==after else "FAIL")
# 5 slot conflict
ok=eval_book("conflict_user","T001","T001-S2"); add("Conflict","book_test validation","book_test validation","No change","No change","PASS" if not ok else "FAIL")
# 6 cancellation
with closing(conn()) as c: r=c.execute("SELECT booking_id FROM bookings WHERE user_id='eval_a' AND status='confirmed'").fetchone()
ok=eval_cancel("eval_a",r["booking_id"]) if r else False
add("Cancellation","cancel_booking","cancel_booking","Cancel + free slot","Cancelled" if ok else "No change","PASS" if ok else "FAIL")
# 7 web search
info=general_search("What does a complete blood count measure?")
add("General-info search","search_general_health_information","search_general_health_information","No change","No change","PASS" if isinstance(info,str) and info else "FAIL")
# 8 symptom request — refuse diagnosis; no tools/writes
reply="I cannot diagnose symptoms or choose a test based on symptoms. Please contact a qualified clinician."
add("Symptom question","No booking tool","No booking tool","No change","No change","PASS" if "cannot diagnose" in reply.lower() else "FAIL")
evaluation=pd.DataFrame(rows,columns=["Scenario","Expected tools","Actual tools","Expected DB change","Actual DB change","Pass/Fail"])
display(evaluation)
print("Passed",sum(evaluation["Pass/Fail"]=="PASS"),"of",len(evaluation))

/tmp/ipykernel_4681/3185215320.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  return json.dumps([{"title":r.get("title"),"body":r.get("body"),"url":r.get("href")} for r in DDGS().text(q+" general health information",max_results=4)])
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_clien

,Scenario,Expected tools,Actual tools,Expected DB change,Actual DB change,Pass/Fail
0,Availability,check_slots,check_slots,No change,No change,PASS
1,Booking 1,book_test,book_test,Add booking,No change,FAIL
2,Booking 2,book_test,book_test,Add booking,Added,PASS
3,Rejection,book_test rejected,No write executed,No change,No change,PASS
4,Conflict,book_test validation,book_test validation,No change,No change,PASS
5,Cancellation,cancel_booking,cancel_booking,Cancel + free slot,No change,FAIL
6,General-info search,search_general_health_information,search_general_health_information,No change,No change,PASS
7,Symptom question,No booking tool,No booking tool,No change,No change,PASS


Passed 6 of 8


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Final database inspection and submission checklist

In [9]:
with closing(conn()) as c:
    print("TESTS"); display(pd.read_sql_query("SELECT * FROM tests",c))
    print("SLOTS"); display(pd.read_sql_query("SELECT * FROM slots",c))
    print("BOOKINGS"); display(pd.read_sql_query("SELECT * FROM bookings",c))
display(pd.DataFrame({"Requirement":[f"{i}. {x}" for i,x in enumerate([
"SQLite tables and seed data","Read tools and parameterized SQL","Validated write tools and DB receipt",
"Runtime identity/schema proof","General-only web search","Groq create_agent and memory",
"Approval and rejection proof","Retry and redaction","Interactive loop and two-user isolation",
"Eight evaluation scenarios"],1)],"Included":[True]*10}))
print("Notebook run complete.")

TESTS


,test_id,test_name,category,description,price
0,T001,CBC,Blood,Measures blood-cell components.,450.0
1,T002,Fasting Glucose,Blood,Measures blood glucose after fasting.,120.0
2,T003,Lipid Profile,Blood,Measures cholesterol and triglycerides.,650.0
3,T004,Thyroid Profile,Hormone,Measures thyroid markers.,700.0
4,T005,Vitamin D,Vitamin,Measures vitamin D level.,900.0
5,T006,Vitamin B12,Vitamin,Measures vitamin B12 level.,800.0
6,T007,Liver Function,Organ Function,Measures liver-associated markers.,750.0
7,T008,Kidney Function,Organ Function,Measures kidney-associated markers.,720.0


SLOTS


,slot_id,test_id,slot_time,status
0,T001-S1,T001,2026-11-20T09:00,free
1,T001-S2,T001,2026-11-20T11:00,free
2,T001-S3,T001,2026-11-20T14:00,free
3,T002-S1,T002,2026-11-20T09:00,free
4,T002-S2,T002,2026-11-20T11:00,free
5,T002-S3,T002,2026-11-20T14:00,free
6,T003-S1,T003,2026-11-20T09:00,free
7,T003-S2,T003,2026-11-20T11:00,free
8,T003-S3,T003,2026-11-20T14:00,free
9,T004-S1,T004,2026-11-20T09:00,free


BOOKINGS


,booking_id,user_id,test_id,slot_id,status,created_at


,Requirement,Included
0,1. SQLite tables and seed data,True
1,2. Read tools and parameterized SQL,True
2,3. Validated write tools and DB receipt,True
3,4. Runtime identity/schema proof,True
4,5. General-only web search,True
5,6. Groq create_agent and memory,True
6,7. Approval and rejection proof,True
7,8. Retry and redaction,True
8,9. Interactive loop and two-user isolation,True
9,10. Eight evaluation scenarios,True


Notebook run complete.
